In [4]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import contextily as cx
from adjustText import adjust_text
import itertools


from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.nh_run import continue_run
from neuralhydrology.utils.config import Config
from neuralhydrology.evaluation import get_tester, metrics

In [5]:
# -------- Paths ---------
CONFIG_PATH = Path("./531_caravan.yml")
RUNS_DIR = Path("runs")

In [6]:
# # by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
# if torch.cuda.is_available() or torch.backends.mps.is_available():
#     start_run(config_file=Path("531_caravan.yml"))

# # fall back to CPU-only mode
# else:
#     start_run(config_file=Path("531_caravan.yml"), gpu=-1)

In [7]:
precip_products = [
    # "camels_precipitation",
    "total_precipitation_sum",
]

base_non_precip_inputs = [
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
]

with open(CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()

seeds = [222, 333, 444, 555, 666] #, 777, 888 

for precip in precip_products:
    for seed in seeds:
        config = base_config.copy()
        config["dynamic_inputs"] = [*base_non_precip_inputs, precip]
        config["seed"] = seed
        config["experiment_name"] = (
            f"{precip}_seq_{config['seq_length']}"
            f"_{config['predict_last_n']}_epochs_{config['epochs']}"
            f"_hidden_{config['hidden_size']}"
            f"_dropout_{str(config['output_dropout']).replace('.', '')}"
            f"_fb_{config['initial_forget_bias']}"
            f"_seed{seed}"
        )

        temp_config_path = Path(f"temp_{precip}_seed{seed}.yml")
        with open(temp_config_path, "w") as f:
            yaml.dump(config, f)

        print(f"Running: {config['experiment_name']}")

        if use_gpu:
            start_run(config_file=temp_config_path)
        else:
            start_run(config_file=temp_config_path, gpu=-1)

        temp_config_path.unlink()

Running: total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222
2026-04-24 23:00:20,546: Logging to /home/azureuser/sky_workdir/final_runs/531_caravan_vs_camels/runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2404_230020/output.log initialized.
2026-04-24 23:00:20,547: ### Folder structure created at /home/azureuser/sky_workdir/final_runs/531_caravan_vs_camels/runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2404_230020
2026-04-24 23:00:20,547: ### Run configurations for total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222
2026-04-24 23:00:20,548: batch_size: 256
2026-04-24 23:00:20,549: clip_gradient_norm: 1
2026-04-24 23:00:20,549: data_dir: /inputs/data_updated_2
2026-04-24 23:00:20,550: dataset: generic
2026-04-24 23:00:20,550: device: cuda:0
2026-04-24 23:00:20,551: dynamic_inputs: ['temperature_2m_max', 'temperature_2m_min', 'surface_net_solar_radiation_mean', 

2026-04-24 23:00:21,512: Loading basin data into xarray data set.
100%|██████████| 531/531 [07:11<00:00,  1.23it/s]
2026-04-24 23:07:33,341: Calculating target variable stds per basin
100%|██████████| 531/531 [00:00<00:00, 1872.89it/s]
2026-04-24 23:07:33,689: Create lookup table and convert to pytorch tensor
100%|██████████| 531/531 [00:13<00:00, 38.99it/s]
2026-04-24 23:07:47,527: Validation set to validate every 1 epoch(s), but 'validate_n_random_basins' not set or set to zero. Will validate on the entire validation set.
# Epoch 1: 100%|██████████| 6821/6821 [06:16<00:00, 18.11it/s, Loss: 0.0233]
2026-04-24 23:14:04,251: Epoch 1 average loss: avg_loss: 0.04599, avg_total_loss: 0.04599
# Validation: 100%|██████████| 531/531 [17:41<00:00,  2.00s/it]
2026-04-24 23:31:46,256: Stored metrics at /home/azureuser/sky_workdir/final_runs/531_caravan_vs_camels/runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2404_230020/validation/model_epoch001/validation_me